# SAM Predictions vs Ground Truth Annotations

This notebook evaluates Segment Anything Model (SAM) predictions against COCO ground truth masks.

## Evaluation Metrics:
- **IoU (Intersection over Union)**: Overlap between predicted and ground truth masks
- **Dice Coefficient**: Similarity measure for segmentation
- **Precision & Recall**: Per-pixel classification accuracy
- **Boundary F1 Score**: Edge quality metric
- **Mean metrics across dataset samples**

## 1. Install Dependencies

In [9]:
# from code.MaskRefiner import MaskRefiner
!pip install segment-anything git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python matplotlib pycocotools scikit-image pandas seaborn

  Cloning https://github.com/facebookresearch/segment-anything.git to /private/var/folders/gx/5qw2j36n0d14_dkv7wr87tfw0000gn/T/pip-req-build-luik4wt3
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /private/var/folders/gx/5qw2j36n0d14_dkv7wr87tfw0000gn/T/pip-req-build-luik4wt3
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Import Libraries

In [10]:
import torch
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import urllib.request
import os
import pandas as pd
from tqdm import tqdm
import json
from coding.FunctionUtils import load_sa1b_image_and_annotations
# from google.colab import drive
# drive.mount('/content/drive')

In [11]:
image_dir = '/Users/carricarte/PhD/Projects/MARL-med/scratch/dataset'
output_dir = '/Users/carricarte/PhD/Projects/MARL-med/scratch/dataset'
file_path = "/Users/carricarte/PhD/Projects/MARL-med/scratch"
# output_dir = '/content/drive/MyDrive/marl/'
threshold = 0.815 # Threshold for challenging segmentations
files = []
[files.append(os.path.join(file_path, f)) for f in os.listdir(file_path) if f.endswith(".csv") and "._" not in f]

[None]

## 3. Download SAM Model

In [12]:
# !nvidia-smi
# Choose model size: 'vit_h' (best), 'vit_l', or 'vit_b' (fastest)
model_type = "vit_b"

checkpoint_url = {
    'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
    'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
    'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
}

checkpoint_path = f"sam_{model_type}.pth"

if not os.path.exists(checkpoint_path):
    print(f"Downloading {model_type} checkpoint...")
    urllib.request.urlretrieve(checkpoint_url[model_type], checkpoint_path)
    print("Download complete!")

# Initialize SAM
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
sam.to(device=device)
predictor = SamPredictor(sam)

print("SAM model loaded!")

Using device: cpu
SAM model loaded!


In [13]:
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()  # ← Actually calls the method
    return hook  # ← Returns the hook function


for i, block in enumerate(sam.image_encoder.blocks):
    block.register_forward_hook(get_activation(f'block_{i}'))

In [14]:
def save_masks_and_embeddings(all_files):

    box_batch_size = 32
    file_no = 1

    for file in tqdm(all_files):

        try:
            df = pd.read_csv(file)

            df_challenging_mask = df[df['IoU'] <= threshold]
            all_pred_masks = []

            for idx, filename in enumerate(tqdm(df_challenging_mask['image_file'].unique())):

                filename = os.path.splitext(filename)[0]
                json_file = os.path.join(image_dir, filename + '.json')

                if os.path.exists(json_file):

                    # print(json_file)
                    # Load image and annotations
                    image, data = load_sa1b_image_and_annotations(json_file)

                    predictor.set_image(image)

                    annotations = pd.DataFrame(data['annotations'])
                    annotations = annotations[annotations['id'].isin(df_challenging_mask['annotation'])]
                    # gt_masks = [decode_sa1b_mask(ann) for ann in annotations]
                    boxes = np.array([[ann['bbox'][0], ann['bbox'][1],
                                  ann['bbox'][0] + ann['bbox'][2],
                                  ann['bbox'][1] + ann['bbox'][3]]
                                 for _, ann in annotations.iterrows()])

                    # Process boxes in batches

                    for batch_start in range(0, len(boxes), box_batch_size):
                        batch_end = min(batch_start + box_batch_size, len(boxes))
                        batch_boxes = boxes[batch_start:batch_end]

                        with torch.no_grad():
                            # Process one box at a time within batch
                            batch_masks = []
                            for a, box in enumerate(batch_boxes):
                                masks, _, _ = predictor.predict(
                                    box=box[None, :],  # Add batch dimension
                                        multimask_output=False)

                                batch_masks.append({"image": filename, "annotation": annotations["id"].iloc[a], "mask":[masks[0]]})

                            all_pred_masks.extend(batch_masks)

                if (idx + 1) % 100 == 0:

                    # Extract annotations and masks
                    images = [item['image'] for item in all_pred_masks]
                    annotations = [item['annotation'] for item in all_pred_masks]
                    masks = [item['mask'] for item in all_pred_masks]

                    mask_arr = np.empty(len(masks), dtype=object)
                    for i, m in enumerate(masks):
                        mask_arr[i] = m   # m is [np.ndarray], stored as-is

                    np.savez_compressed(
                        f'{output_dir}/challenging_masks_{file_no:06d}.npz',
                        image=images,
                        annotation=annotations,
                        mask=mask_arr
                    )
                    print(f"Checkpoint saved at {idx + 1} images, file {file_no}")
                    all_pred_masks = []

                block_name = "block_10"
                block_embedding = activations[block_name]

                flattened_embedding = block_embedding.flatten().detach().cpu().numpy()

                np.save(
                    os.path.join(output_dir, f"{filename}_embeddings_{block_name}.npy"),
                    flattened_embedding
                )

        except Exception as e:

            print(f"Error: {file}: {e}")
            continue

# Run evaluation
print("Starting Refiner training on challenging SA-1B masks...")

df = save_masks_and_embeddings(files)

Starting Refiner training on challenging SA-1B masks...


  0%|          | 0/1 [00:00<?, ?it/s]Error while evaluating expression: idx == 89
Traceback (most recent call last):
  File "/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev/_pydevd_bundle/pydevd_utils.py", line 670, in eval_expression
    return eval(expression, globals, locals)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'idx' is not defined. Did you mean: 'id'?

100%|██████████| 1/1 [01:03<00:00, 63.65s/it]
